# GastroNet — NB1: CNN-only baseline (EfficientNet-B4)

**Prerequisite: NB0 must already have been run on this account** (or its
`dataset_split.json` + `checkpoint_utils.py` copied in from the account that ran it).
This notebook does not generate the split or define checkpoint functions — it
only imports and uses them.

This notebook trains the CNN-only branch (`model_family = "cnn_only"`) as one
of the ablation baselines, on seed 42 (single seed is enough for a baseline —
see NB0's plan notes on why multi-seed is reserved for the models actually
being compared in the paper).

**v2 changes**: added a local-disk copy step (Cell after checkpoint_utils import)
to avoid Google Drive API rate-limiting during training, remapped dataset paths
to that local copy, fixed the `torch.cuda.amp.autocast` deprecation warning, and
**v3 changes**: added early stopping (`PATIENCE=6` epochs without val_acc
improvement) to both the single-seed training loop and a new multi-seed
runner cell (seeds 123 and 7), plus a final cell that aggregates mean±std
test accuracy across every seed found on disk for this model_family.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os, sys, json

# These four must be IDENTICAL to what you set in NB0 on this account.
ACCOUNT_TAG      = "acct_A"
EXPERIMENTS_ROOT = "/content/drive/MyDrive/gastronet_experiments"
NOTEBOOK_NAME    = "NB1_cnn_only_baseline"

# Must be IDENTICAL to what you set in NB0 -- this is where the ORIGINAL
# images live on Drive; the local-copy cell below reads from this path.
RAW_DATASET_DIR = "/content/drive/MyDrive/gastronet_raw_dataset"

# This notebook's own identity within the experiment structure.
MODEL_FAMILY = "cnn_only_v2"
SEED = 42

CLASS_NAMES = ["Diverticulosis", "Neoplasm", "Peritonitis", "Ureters"]
IMG_SIZE = 448
BATCH_SIZE = 16          # T4-safe for EfficientNet-B4 at 448x448; lower to 8 if you hit OOM
NUM_EPOCHS = 30          # ceiling -- resume logic means you can stop/restart across sessions freely
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

SPLIT_JSON_PATH    = os.path.join(EXPERIMENTS_ROOT, "dataset_split_v2.json")
MANIFEST_JSON_PATH = os.path.join(EXPERIMENTS_ROOT, "experiments_manifest.json")

assert os.path.exists(SPLIT_JSON_PATH), (
    "dataset_split.json not found. Run NB0 on this account first, or copy "
    "dataset_split.json + checkpoint_utils.py in from the account that did."
)
assert os.path.exists(os.path.join(EXPERIMENTS_ROOT, "checkpoint_utils.py")), (
    "checkpoint_utils.py not found in EXPERIMENTS_ROOT. Same fix as above."
)

sys.path.insert(0, EXPERIMENTS_ROOT)
import checkpoint_utils as cku
print("checkpoint_utils imported OK from:", EXPERIMENTS_ROOT)


checkpoint_utils imported OK from: /content/drive/MyDrive/gastronet_experiments


### Why this cell exists
Reading thousands of individual image files directly off a mounted Google
Drive during training can hit Drive's API rate limits partway through an
epoch, causing training to slow down 10-15x and stay slow for the rest of
the session. Copying the dataset to local Colab disk once per session avoids
this entirely. This copy is temporary (wiped when the session ends/disconnects)
and needs to happen again after every fresh Colab connect -- it does NOT need
to happen every epoch, and it does NOT touch `dataset_split.json` or
`RAW_DATASET_DIR` on Drive, both of which remain the canonical source.


In [3]:
import shutil

LOCAL_DATASET_DIR = "/content/gastro_local_copy"

if not os.path.exists(LOCAL_DATASET_DIR):
    print("Copying dataset from Drive to local disk (one-time per session)...")
    shutil.copytree(RAW_DATASET_DIR, LOCAL_DATASET_DIR)
    print("Done.")
else:
    print("Local copy already exists this session, skipping copy.")


Copying dataset from Drive to local disk (one-time per session)...
Done.


In [4]:
with open(SPLIT_JSON_PATH) as f:
    split_payload = json.load(f)

SPLIT_HASH = split_payload["split_hash"]
split = split_payload["split"]

assert split_payload["class_names"] == CLASS_NAMES, "Class name mismatch with locked split!"

print("Loaded split_hash:", SPLIT_HASH)
for k in ["train", "val", "test"]:
    print(f"  {k}: {len(split[k])} images")

# Remap paths to the local copy for actual file reads during training.
# dataset_split.json itself is untouched -- it still stores canonical Drive
# paths, and SPLIT_HASH above was computed from those, so hash checks
# elsewhere are unaffected by this in-memory remap.
def remap_to_local(entries):
    return [(p.replace(RAW_DATASET_DIR, LOCAL_DATASET_DIR), cls) for p, cls in entries]

split = {k: remap_to_local(v) for k, v in split.items()}
print("Paths remapped to local copy, e.g.:", split["train"][0][0])


Loaded split_hash: d6e80caa29bff18856ae93ea635b4650
  train: 3198 images
  val: 400 images
  test: 400 images
Paths remapped to local copy, e.g.: /content/gastro_local_copy/Diverticulosis/08648b7f-37b9-406e-b1c3-03683c3222af.jpg


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}

class GastroDataset(Dataset):
    def __init__(self, entries, transform):
        # entries: list of (path, class_name) exactly as stored in dataset_split.json
        self.entries = entries
        self.transform = transform

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        path, cls = self.entries[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        label = class_to_idx[cls]
        return img, label


IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = GastroDataset(split["train"], train_transform)
val_ds   = GastroDataset(split["val"], eval_transform)
test_ds  = GastroDataset(split["test"], eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} batches/epoch={len(train_loader)}")


Device: cuda
train=3198 val=400 test=400 batches/epoch=200


In [6]:
import torch.nn as nn
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

def build_cnn_model(num_classes=len(CLASS_NAMES)):
    weights = EfficientNet_B4_Weights.IMAGENET1K_V1
    model = efficientnet_b4(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

torch.manual_seed(SEED)
model = build_cnn_model().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
criterion = nn.CrossEntropyLoss()

print(model.__class__.__name__, "built, classifier head ->", CLASS_NAMES)


Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [00:00<00:00, 196MB/s]


EfficientNet built, classifier head -> ['Diverticulosis', 'Neoplasm', 'Peritonitis', 'Ureters']


/tmp/ipykernel_1117/4265502673.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


In [7]:
exp_dir = cku.get_experiment_dir(EXPERIMENTS_ROOT, MODEL_FAMILY, SEED)
print("Experiment dir:", exp_dir)

start_epoch, best_val_acc, history = cku.resume_or_start(
    exp_dir, model, optimizer, scheduler, scaler, map_location=device
)

cku.log_run_event(
    MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
    status="resumed" if start_epoch > 0 else "started",
    best_val_acc=best_val_acc, drive_path=exp_dir,
)


Experiment dir: /content/drive/MyDrive/gastronet_experiments/cnn_only_v2/seed_42
[/content/drive/MyDrive/gastronet_experiments/cnn_only_v2/seed_42] No latest.pt found -- starting fresh at epoch 0.


In [8]:
import time

def run_epoch(loader, train_mode, print_every=20):
    model.train() if train_mode else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    start_time = time.time()
    last_print_time = start_time

    with torch.set_grad_enabled(train_mode):
        for batch_idx, (imgs, labels) in enumerate(loader):
            imgs, labels = imgs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            if train_mode:
                optimizer.zero_grad()

            with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
                outputs = model(imgs)
                loss = criterion(outputs, labels)

            if train_mode:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)

            if (batch_idx + 1) % print_every == 0:
                now = time.time()
                chunk_time = now - last_print_time
                last_print_time = now
                running_acc = correct / total
                mode_str = "train" if train_mode else "val"
                print(f"    [{mode_str}] batch {batch_idx+1}/{len(loader)} "
                      f"| running_acc={running_acc:.4f} "
                      f"| this_chunk={chunk_time:.1f}s | total_elapsed={now - start_time:.1f}s")

    return total_loss / total, correct / total


In [9]:
PATIENCE = 6                      # epochs without val_acc improvement before stopping early
epochs_without_improvement = 0

try:
    for epoch in range(start_epoch, NUM_EPOCHS):
        train_loss, train_acc = run_epoch(train_loader, train_mode=True)
        val_loss, val_acc = run_epoch(val_loader, train_mode=False)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        # ALWAYS save latest.pt, every epoch -- this is what survives a Colab disconnect
        cku.save_latest(exp_dir, epoch, model, optimizer, scheduler, scaler, best_val_acc, history)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_without_improvement = 0
            config = {
                "model_family": MODEL_FAMILY,
                "seed": SEED,
                "split_hash": SPLIT_HASH,
                "account_tag": ACCOUNT_TAG,
                "notebook_name": NOTEBOOK_NAME,
                "img_size": IMG_SIZE,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "class_names": CLASS_NAMES,
            }
            cku.save_best(exp_dir, model, epoch, best_val_acc, config)
            cku.save_config(exp_dir, config)
            print(f"  -> new best_val_acc={best_val_acc:.4f}, saved best.pt")
        else:
            epochs_without_improvement += 1
            print(f"  -> no improvement for {epochs_without_improvement}/{PATIENCE} epochs")
            if epochs_without_improvement >= PATIENCE:
                print(f"Stopping early: no val_acc improvement for {PATIENCE} epochs.")
                cku.save_history(exp_dir, history)
                break

        cku.save_history(exp_dir, history)

    cku.log_run_event(
        MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
        status="completed", best_val_acc=best_val_acc, drive_path=exp_dir,
    )
    print(f"\nTraining loop finished. best_val_acc={best_val_acc:.4f}")

except KeyboardInterrupt:
    cku.log_run_event(
        MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
        status="interrupted", best_val_acc=best_val_acc, drive_path=exp_dir,
        note="Manually interrupted -- latest.pt has the current state, safe to resume.",
    )
    print("Interrupted -- latest.pt is up to date through the last completed epoch. "
          "Just re-run this notebook from Cell 3 onward to resume.")


    [train] batch 20/200 | running_acc=0.3500 | this_chunk=24.0s | total_elapsed=24.0s
    [train] batch 40/200 | running_acc=0.4500 | this_chunk=8.0s | total_elapsed=32.1s
    [train] batch 60/200 | running_acc=0.5542 | this_chunk=8.6s | total_elapsed=40.7s
    [train] batch 80/200 | running_acc=0.6117 | this_chunk=7.8s | total_elapsed=48.5s
    [train] batch 100/200 | running_acc=0.6444 | this_chunk=8.0s | total_elapsed=56.5s
    [train] batch 120/200 | running_acc=0.6854 | this_chunk=8.2s | total_elapsed=64.7s
    [train] batch 140/200 | running_acc=0.7188 | this_chunk=7.7s | total_elapsed=72.4s
    [train] batch 160/200 | running_acc=0.7426 | this_chunk=8.3s | total_elapsed=80.8s
    [train] batch 180/200 | running_acc=0.7625 | this_chunk=7.8s | total_elapsed=88.5s
    [train] batch 200/200 | running_acc=0.7780 | this_chunk=21.2s | total_elapsed=109.8s
    [val] batch 20/25 | running_acc=0.9688 | this_chunk=4.6s | total_elapsed=4.6s
Epoch 1/30 | train_loss=0.7297 train_acc=0.7780 |

In [10]:
best_ckpt = cku.load_best(exp_dir, map_location=device)
assert best_ckpt is not None, "No best.pt found -- training must complete at least one improving epoch first."

cku.assert_split_hash_matches(best_ckpt["config"], SPLIT_HASH)

eval_model = build_cnn_model().to(device)
eval_model.load_state_dict(best_ckpt["model_state_dict"])
eval_model.eval()

correct, total = 0, 0
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = eval_model(imgs)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

test_acc = correct / total
print(f"Test accuracy ({MODEL_FAMILY}, seed {SEED}): {test_acc:.4f}  (best.pt was from epoch {best_ckpt['epoch']})")

results = {
    "model_family": MODEL_FAMILY,
    "seed": SEED,
    "split_hash": SPLIT_HASH,
    "account_tag": ACCOUNT_TAG,
    "best_epoch": best_ckpt["epoch"],
    "best_val_acc": best_ckpt["best_val_acc"],
    "test_accuracy": test_acc,
    "predictions": all_preds,
    "labels": all_labels,
    "class_names": CLASS_NAMES,
}
cku.save_results(exp_dir, results)
print("results.json saved to:", exp_dir)


Test accuracy (cnn_only_v2, seed 42): 0.9825  (best.pt was from epoch 3)
results.json saved to: /content/drive/MyDrive/gastronet_experiments/cnn_only_v2/seed_42


In [11]:
cku.manifest_summary(MANIFEST_JSON_PATH)


_setup             seed_0      -> completed    acc=None account=acct_A notebook=NB0_setup_and_split_lock at 2026-09-07 10:15:46
cnn_only           seed_7      -> completed    acc=0.985 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:56:38
cnn_only           seed_42     -> resumed      acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:26:57
cnn_only           seed_123    -> completed    acc=0.9825 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-06 12:42:24
cnn_only_v2        seed_42     -> completed    acc=0.985 account=acct_A notebook=NB1_cnn_only_baseline at 2026-09-07 11:01:48


## Multi-seed runs (seeds 123 and 7)

Run this cell only after seed 42 above is finished and its `results.json` looks
good. It loops over the remaining seeds, training/evaluating each one fully
before moving to the next, with the same early-stopping and checkpoint
behavior as the single-seed cells above. It skips any seed that already has a
`results.json` (so it's safe to re-run this cell if a session disconnects
partway through the seed list), and deletes each seed's `latest.pt` the
moment that seed is confirmed done, so at most one large checkpoint file
exists on Drive at a time.


In [12]:
SEEDS_TO_RUN = [123, 7]   # seed 42 already done above, not repeated here
PATIENCE = 6

all_seed_results = {}

for SEED in SEEDS_TO_RUN:
    seed_exp_dir = cku.get_experiment_dir(EXPERIMENTS_ROOT, MODEL_FAMILY, SEED)

    if os.path.exists(os.path.join(seed_exp_dir, "results.json")):
        print(f"Seed {SEED} already has results.json, skipping.")
        with open(os.path.join(seed_exp_dir, "results.json")) as f:
            all_seed_results[SEED] = json.load(f)
        continue

    print(f"\n{'='*60}\nStarting SEED={SEED}\n{'='*60}")

    torch.manual_seed(SEED)
    seed_model = build_cnn_model().to(device)
    seed_optimizer = torch.optim.AdamW(seed_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    seed_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(seed_optimizer, T_max=NUM_EPOCHS)
    seed_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    seed_start_epoch, seed_best_val_acc, seed_history = cku.resume_or_start(
        seed_exp_dir, seed_model, seed_optimizer, seed_scheduler, seed_scaler, map_location=device
    )
    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="resumed" if seed_start_epoch > 0 else "started",
                       best_val_acc=seed_best_val_acc, drive_path=seed_exp_dir)

    epochs_without_improvement = 0

    # Temporarily point the shared train/eval globals at this seed's objects,
    # since run_epoch() (Cell 8) closes over model/optimizer/scaler by name.
    model, optimizer, scheduler, scaler = seed_model, seed_optimizer, seed_scheduler, seed_scaler

    for epoch in range(seed_start_epoch, NUM_EPOCHS):
        train_loss, train_acc = run_epoch(train_loader, train_mode=True)
        val_loss, val_acc = run_epoch(val_loader, train_mode=False)
        scheduler.step()

        seed_history["train_loss"].append(train_loss)
        seed_history["val_loss"].append(val_loss)
        seed_history["val_acc"].append(val_acc)
        print(f"[seed {SEED}] Epoch {epoch+1}/{NUM_EPOCHS} | "
              f"train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        cku.save_latest(seed_exp_dir, epoch, model, optimizer, scheduler, scaler,
                         seed_best_val_acc, seed_history)

        if val_acc > seed_best_val_acc:
            seed_best_val_acc = val_acc
            epochs_without_improvement = 0
            seed_config = {
                "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
                "account_tag": ACCOUNT_TAG, "notebook_name": NOTEBOOK_NAME,
                "img_size": IMG_SIZE, "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE, "class_names": CLASS_NAMES,
            }
            cku.save_best(seed_exp_dir, model, epoch, seed_best_val_acc, seed_config)
            cku.save_config(seed_exp_dir, seed_config)
            print(f"  -> new best_val_acc={seed_best_val_acc:.4f}, saved best.pt")
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"Stopping early for seed {SEED}: no improvement for {PATIENCE} epochs.")
                cku.save_history(seed_exp_dir, seed_history)
                break

        cku.save_history(seed_exp_dir, seed_history)

    # Test-set evaluation for this seed, same logic as Cell 10
    best_ckpt = cku.load_best(seed_exp_dir, map_location=device)
    cku.assert_split_hash_matches(best_ckpt["config"], SPLIT_HASH)
    eval_model = build_cnn_model().to(device)
    eval_model.load_state_dict(best_ckpt["model_state_dict"])
    eval_model.eval()

    correct, total, preds_all, labels_all = 0, 0, [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = eval_model(imgs).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
            preds_all.extend(preds.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())

    seed_test_acc = correct / total
    seed_results = {
        "model_family": MODEL_FAMILY, "seed": SEED, "split_hash": SPLIT_HASH,
        "account_tag": ACCOUNT_TAG, "best_epoch": best_ckpt["epoch"],
        "best_val_acc": best_ckpt["best_val_acc"], "test_accuracy": seed_test_acc,
        "predictions": preds_all, "labels": labels_all, "class_names": CLASS_NAMES,
    }
    cku.save_results(seed_exp_dir, seed_results)
    all_seed_results[SEED] = seed_results

    cku.log_run_event(MANIFEST_JSON_PATH, MODEL_FAMILY, SEED, ACCOUNT_TAG, NOTEBOOK_NAME,
                       status="completed", best_val_acc=seed_best_val_acc, drive_path=seed_exp_dir)

    # Free this seed's latest.pt immediately -- keeps at most one large
    # checkpoint alive on Drive at any point during the multi-seed loop.
    cku.finalize_experiment(seed_exp_dir)
    print(f"Seed {SEED} done. test_acc={seed_test_acc:.4f}. latest.pt cleaned up.\n")

print("\nAll requested seeds complete:")
for s, r in all_seed_results.items():
    print(f"  seed {s}: test_accuracy={r['test_accuracy']:.4f}, best_val_acc={r['best_val_acc']:.4f}")



Starting SEED=123
[/content/drive/MyDrive/gastronet_experiments/cnn_only_v2/seed_123] No latest.pt found -- starting fresh at epoch 0.


/tmp/ipykernel_1117/1976684885.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  seed_scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))


    [train] batch 20/200 | running_acc=0.3781 | this_chunk=9.3s | total_elapsed=9.3s
    [train] batch 40/200 | running_acc=0.4969 | this_chunk=8.2s | total_elapsed=17.6s
    [train] batch 60/200 | running_acc=0.5865 | this_chunk=8.8s | total_elapsed=26.3s
    [train] batch 80/200 | running_acc=0.6336 | this_chunk=8.4s | total_elapsed=34.7s
    [train] batch 100/200 | running_acc=0.6719 | this_chunk=8.7s | total_elapsed=43.4s
    [train] batch 120/200 | running_acc=0.7089 | this_chunk=8.3s | total_elapsed=51.7s
    [train] batch 140/200 | running_acc=0.7353 | this_chunk=8.3s | total_elapsed=59.9s
    [train] batch 160/200 | running_acc=0.7590 | this_chunk=8.3s | total_elapsed=68.2s
    [train] batch 180/200 | running_acc=0.7785 | this_chunk=8.0s | total_elapsed=76.3s
    [train] batch 200/200 | running_acc=0.7927 | this_chunk=8.4s | total_elapsed=84.7s
    [val] batch 20/25 | running_acc=0.9594 | this_chunk=4.3s | total_elapsed=4.3s
[seed 123] Epoch 1/30 | train_acc=0.7927 val_acc=0.95

In [13]:
import numpy as np
import glob

all_results = []
for rf in sorted(glob.glob(os.path.join(EXPERIMENTS_ROOT, MODEL_FAMILY, "seed_*", "results.json"))):
    with open(rf) as f:
        all_results.append(json.load(f))

test_accs = [r["test_accuracy"] for r in all_results]
seeds_found = [r["seed"] for r in all_results]

print(f"{MODEL_FAMILY} -- seeds found: {seeds_found}")
print(f"Test accuracies: {[round(a, 4) for a in test_accs]}")
if len(test_accs) > 1:
    print(f"Mean: {np.mean(test_accs):.4f}  Std: {np.std(test_accs, ddof=1):.4f}")
else:
    print("Only one seed found so far -- run the multi-seed cell above to get mean/std.")


cnn_only_v2 -- seeds found: [123, 42, 7]
Test accuracies: [0.9775, 0.9825, 0.98]
Mean: 0.9800  Std: 0.0025


## After this notebook finishes

0. **If you see a chunk in the per-batch prints suddenly take much longer than previous chunks** (e.g. `this_chunk=9s` jumping to `this_chunk=120s`) and it doesn't recover within one more print interval, interrupt (Runtime → Interrupt), confirm the local-copy cell ran successfully this session, and re-run from Cell 3. Nothing is lost — `latest.pt`/`best.pt` only get written at the end of a completed epoch.
1. **Check `results.json` and `history.json`** in `gastronet_experiments/cnn_only/seed_42/` — confirm `test_accuracy` and `best_val_acc` look sane (not near-chance, not suspiciously perfect).
2. **Do not call `finalize_experiment()` from inside a code cell automatically.** Once you're satisfied `best.pt` + `results.json` are correct, run this manually in a scratch cell:
   ```python
   cku.finalize_experiment(exp_dir)
   ```
   This deletes `latest.pt` (the large resumable file) and keeps only the permanent `best.pt` — this is the point where you actually reclaim Drive space for this experiment.
3. **If a Colab session disconnects mid-training**: just re-open this notebook and re-run from Cell 3 onward. `resume_or_start` will pick up exactly where `latest.pt` left off — you never lose more than the current epoch's partial progress.
4. **Multi-seed**: once seed 42 looks good, run the "Multi-seed runs" cell to train seeds 123 and 7 automatically, one after another, with the same early-stopping and checkpointing behavior. It's safe to re-run if interrupted — it skips any seed that already has a `results.json`.
5. **Mean ± std**: run the final aggregation cell any time to see mean/std test accuracy across whatever seeds have finished so far.
6. **Next notebook: NB2 (ViT-only baseline).** Same structure as this one, with `MODEL_FAMILY = "vit_only"` and a ViT-Small backbone swapped in for Cell 6 — everything else (split loading, resume, early stopping, multi-seed, manifest logging, finalize) is copy-identical because it all goes through `checkpoint_utils`.
